# Glia RAG Cache — Enhanced Demo

**Start Redis first:**
```bash
docker compose up -d
```

**Install dependencies:**
```bash
pip install glia sentence-transformers redis redisvl groq numpy
```

### What this notebook covers
| Section | What you'll see |
|---|---|
| 1 | Imports & config |
| 2 | In-memory Vector DB with real embeddings |
| 3 | Upload documents → embed → store in Vector DB |
| 4 | Init GliaManager + wire VectorDB → Redis cache |
| 5 | Ask questions — clear CACHE HIT / MISS banner |
| 6 | Update a document in the Vector DB → auto-invalidate cache |
| 7 | Inspect Redis cache contents |
| 8 | Full cleanup |

## 1. Imports & Configuration

In [1]:
import os
import json
import uuid
import time
import warnings
import numpy as np
from datetime import datetime
from copy import deepcopy
from typing import List, Dict, Any, Optional

warnings.filterwarnings("ignore")

from groq import Groq
from sentence_transformers import SentenceTransformer

from glia.manager import GliaManager
from glia.invalidator import CacheInvalidator

print("Imports OK")

Imports OK


In [ ]:
# ── Connection & model config ────────────────────────────────────────
REDIS_URL    = "redis://localhost:6379"
INDEX_NAME   = "rag_cache"
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "YOUR_GROQ_API_KEY_HERE")
GROQ_MODEL   = "openai/gpt-oss-20b"

print(f"Redis  : {REDIS_URL}")
print(f"Index  : {INDEX_NAME}")
print(f"Model  : {GROQ_MODEL}")

Redis  : redis://localhost:6379
Index  : rag_cache
Model  : openai/gpt-oss-20b


## 2. In-Memory Vector Database

A lightweight vector store backed by numpy cosine similarity.  
It mirrors the interface that `VectorDBAdapter` expects:
- `fetch_updated(collection, since, timestamp_field)` → polling
- `describe_collection(collection)` → liveness probe

Every document is stored as:
```python
{
  "doc_id"     : str,          # stable identifier → source_id in Redis cache
  "text"       : str,          # original text content
  "embedding"  : List[float],  # sentence-transformer vector
  "updated_at" : datetime,     # watermark for polling
  "metadata"   : dict,         # optional extra fields
}
```

In [3]:
class InMemoryVectorDB:
    """
    Minimal in-process vector store with numpy cosine similarity search.
    Exposes the interface expected by VectorDBAdapter (polling mode).
    """

    def __init__(self):
        # collection_name → list of document dicts
        self._store: Dict[str, List[Dict]] = {}

    # ── CRUD ────────────────────────────────────────────────────────────

    def upsert(self, collection: str, doc: Dict) -> None:
        """Insert or overwrite a document by doc_id."""
        if collection not in self._store:
            self._store[collection] = []
        # Remove existing entry with the same doc_id (upsert semantics)
        self._store[collection] = [
            d for d in self._store[collection] if d["doc_id"] != doc["doc_id"]
        ]
        doc["updated_at"] = datetime.utcnow()
        self._store[collection].append(doc)

    def get(self, collection: str, doc_id: str) -> Optional[Dict]:
        """Return a document by doc_id, or None."""
        for doc in self._store.get(collection, []):
            if doc["doc_id"] == doc_id:
                return doc
        return None

    def delete(self, collection: str, doc_id: str) -> bool:
        """Delete a document; return True if it existed."""
        before = len(self._store.get(collection, []))
        self._store[collection] = [
            d for d in self._store.get(collection, []) if d["doc_id"] != doc_id
        ]
        return len(self._store.get(collection, [])) < before

    def list_all(self, collection: str) -> List[Dict]:
        """Return all documents in a collection."""
        return list(self._store.get(collection, []))

    # ── Similarity search ───────────────────────────────────────────────

    def search(
        self,
        collection: str,
        query_vector: List[float],
        top_k: int = 3,
    ) -> List[Dict]:
        """
        Return the top-k most similar documents to `query_vector`
        using cosine similarity.
        """
        docs = self._store.get(collection, [])
        if not docs:
            return []

        q = np.array(query_vector, dtype=np.float32)
        q /= np.linalg.norm(q) + 1e-9

        scored = []
        for doc in docs:
            emb = np.array(doc["embedding"], dtype=np.float32)
            emb /= np.linalg.norm(emb) + 1e-9
            score = float(np.dot(q, emb))
            scored.append({**doc, "_score": score})

        scored.sort(key=lambda x: x["_score"], reverse=True)
        return scored[:top_k]

    # ── VectorDBAdapter polling interface ───────────────────────────────

    def describe_collection(self, collection: str) -> Dict:
        """Liveness probe used by VectorDBAdapter.connect()."""
        return {"collection": collection, "count": len(self._store.get(collection, []))}

    def fetch_updated(
        self,
        collection: str,
        since=None,
        timestamp_field: str = "updated_at",
    ) -> List[Dict]:
        """Return docs newer than `since` — used by VectorDBAdapter.poll()."""
        docs = self._store.get(collection, [])
        if since is None:
            return list(docs)
        return [d for d in docs if d.get(timestamp_field) and d[timestamp_field] > since]


# Single shared instance used throughout the notebook
vector_db = InMemoryVectorDB()
print("InMemoryVectorDB ready")

InMemoryVectorDB ready


## 3. Vectorizer + Upload Documents to Vector DB

Edit `DOCUMENTS` below to add your own text entries.  
Each entry is a `(doc_id, text)` tuple — `doc_id` becomes the `source_id` tag in Redis.

In [4]:
class STVectorizer:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
        self.dims  = self.model.get_sentence_embedding_dimension()

    def embed(self, text: str) -> List[float]:
        return self.model.encode(text).tolist()

    def embed_many(self, texts: List[str]) -> List[List[float]]:
        return self.model.encode(texts).tolist()


vectorizer = STVectorizer()
print(f"✅ Vectorizer ready  dims={vectorizer.dims}")

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 1659.27it/s]


✅ Vectorizer ready  dims=384


In [5]:
# ── Define your documents here ───────────────────────────────────────
# Each tuple: (doc_id, text_content)
# doc_id becomes the source_id tag that links the Vector DB entry to
# cached responses in Redis — changing the doc invalidates those entries.

COLLECTION = "knowledge_base"

DOCUMENTS = [
    (
        "doc:climate",
        "Climate change is primarily driven by human emissions of greenhouse gases "
        "like CO2 from fossil fuels, deforestation, and industrial processes. "
        "These gases trap heat in the atmosphere, raising global temperatures."
        "Climate change is also reflected in a wide range of measurable environmental indicators beyond greenhouse gas concentrations. Global average surface temperatures have risen significantly since the late 19th century, with the most rapid warming occurring in recent decades. This warming is closely linked to the increased frequency and intensity of extreme weather events such as heatwaves, heavy rainfall, droughts, and tropical storms. Ocean temperatures have also increased, contributing to coral bleaching and the disruption of marine ecosystems, while thermal expansion and melting glaciers and ice sheets have driven a steady rise in sea levels. Arctic sea ice extent has declined sharply, particularly during summer months, reducing the Earth's albedo effect and accelerating warming. Additionally, climate data shows shifts in precipitation patterns, with some regions experiencing more intense rainfall and flooding, while others face prolonged dry periods. Atmospheric CO₂ concentrations have surpassed 420 parts per million in recent years—levels not seen in millions of years—highlighting the unprecedented rate of change. Together, these datasets provide strong, consistent evidence of a warming planet and ongoing climate system disruption.",
    ),
    (
        "doc:python",
        "Python is a high-level, interpreted programming language known for its "
        "clean syntax. It is widely used for web development, data science, "
        "machine learning, automation, and scripting."
        "Python is a high-level, interpreted programming language widely used across many domains due to its readability and versatility. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming, making it suitable for both beginners and experienced developers. Python’s extensive standard library and ecosystem of third-party packages enable applications in web development, data analysis, artificial intelligence, scientific computing, automation, and more. Popular frameworks like Django and Flask are used for building web applications, while libraries such as NumPy, pandas, and TensorFlow support data science and machine learning tasks. Python uses dynamic typing and automatic memory management, which simplifies development and reduces the need for boilerplate code. Its large global community contributes to continuous improvements, abundant documentation, and a wide range of open-source tools, making Python one of the most in-demand and widely adopted programming languages today.",
    ),
    (
        "doc:redis",
        "Redis is an open-source, in-memory data structure store used as a "
        "database, cache, and message broker. It supports strings, hashes, "
        "lists, sets, sorted sets, bitmaps, and more."
        "Redis is an open-source, in-memory data structure store commonly used as a database, cache, and message broker due to its high performance and low latency. Unlike traditional disk-based databases, Redis stores data primarily in RAM, allowing it to process millions of requests per second with sub-millisecond response times. It supports a variety of data structures such as strings, hashes, lists, sets, sorted sets, bitmaps, and hyperloglogs, making it highly flexible for different use cases. Redis also provides features like persistence (RDB snapshots and AOF logs), replication, and clustering for high availability and scalability. It is frequently used for caching frequently accessed data, session storage in web applications, real-time analytics, pub/sub messaging systems, and queue management. Additionally, Redis includes built-in support for transactions, Lua scripting, and expiration of keys, which helps developers build efficient and responsive systems.",
    ),
    (
        "doc:transformers",
        "Transformer neural networks use a self-attention mechanism to process "
        "sequences in parallel. They underpin modern LLMs like GPT and BERT, "
        "excelling at tasks such as translation, summarisation, and question answering."
        "Transformers are a class of deep learning models that have become the foundation of modern natural language processing and many other AI applications. Introduced in the 2017 paper “Attention Is All You Need,” transformers rely on a mechanism called self-attention to process input data, allowing them to weigh the importance of different words or tokens in a sequence regardless of their position. This enables them to capture long-range dependencies more effectively than earlier models like recurrent or convolutional neural networks. Transformers consist of encoder and decoder layers, each made up of attention heads and feedforward neural networks, which work together to generate contextual representations of data. They are highly parallelizable, making them efficient to train on large datasets using modern hardware like GPUs and TPUs. Popular transformer-based models include BERT, GPT, and T5, which are used for tasks such as text classification, translation, summarization, question answering, and code generation. Beyond text, transformers have also been adapted for use in computer vision, speech processing, and multimodal systems, demonstrating their flexibility and broad impact across artificial intelligence.",
    ),
    (
        "doc:rag",
        "Retrieval-Augmented Generation (RAG) enhances LLM responses by retrieving "
        "relevant documents from a knowledge base at query time and injecting them "
        "into the prompt context before generation."
        "Retrieval-Augmented Generation (RAG) is a hybrid approach in artificial intelligence that combines information retrieval with text generation to produce more accurate and context-aware outputs. Instead of relying solely on a model’s internal knowledge, RAG systems first retrieve relevant documents or data from external sources—such as databases, knowledge bases, or vector stores—and then use that information to guide the generation process. This significantly improves factual accuracy, reduces hallucinations, and allows models to access up-to-date or domain-specific knowledge without retraining. A typical RAG pipeline involves embedding user queries, searching for similar content using techniques like vector similarity, and feeding the retrieved context into a generative model (often a transformer-based model) to produce a final response. RAG is widely used in applications like question answering systems, enterprise search, chatbots, and document summarization. It also enables better transparency, since the retrieved sources can be inspected, and offers scalability by allowing systems to update knowledge simply by modifying the underlying data store rather than retraining the entire model.",
    ),
]

# ── Embed and upload ─────────────────────────────────────────────────
print(f"Uploading {len(DOCUMENTS)} documents to collection '{COLLECTION}'…\n")

texts = [text for _, text in DOCUMENTS]
embeddings = vectorizer.embed_many(texts)

for (doc_id, text), embedding in zip(DOCUMENTS, embeddings):
    vector_db.upsert(
        COLLECTION,
        {
            "doc_id":    doc_id,
            "text":      text,
            "embedding": embedding,
            "metadata":  {},
        },
    )
    print(f"  ✅ [{doc_id}]  {text[:70]}…")

print(f"\nTotal docs in '{COLLECTION}': {len(vector_db.list_all(COLLECTION))}")

Uploading 5 documents to collection 'knowledge_base'…

  ✅ [doc:climate]  Climate change is primarily driven by human emissions of greenhouse ga…
  ✅ [doc:python]  Python is a high-level, interpreted programming language known for its…
  ✅ [doc:redis]  Redis is an open-source, in-memory data structure store used as a data…
  ✅ [doc:transformers]  Transformer neural networks use a self-attention mechanism to process …
  ✅ [doc:rag]  Retrieval-Augmented Generation (RAG) enhances LLM responses by retriev…

Total docs in 'knowledge_base': 5


## 4. Initialise GliaManager, CacheInvalidator & Groq

In [7]:
manager = GliaManager(
    vectorizer=vectorizer,
    redis_url=REDIS_URL,
    index_name=INDEX_NAME,
    distance_threshold=0.25,
    vector_dims=vectorizer.dims,
)

invalidator  = CacheInvalidator(cache_manager=manager)
groq_client  = Groq(api_key=GROQ_API_KEY)

print("GliaManager ready")
print("CacheInvalidator ready")
print("Groq client ready")

GliaManager ready
CacheInvalidator ready
Groq client ready


## 5. Ask Questions — Cache Hit / Miss Indicator

The `ask()` helper:
1. Searches the **Vector DB** for the most relevant document.
2. Checks the **Redis semantic cache** — prints a coloured `[CACHE HIT]` or `[CACHE MISS]` banner.
3. On a miss: calls **Groq**, then stores the response in Redis tagged with the `doc_id`.

Semantically similar follow-up questions will hit the cache without calling Groq again.

In [8]:
def _banner(label: str, color_code: str, width: int = 62) -> str:
    bar = "─" * width
    return f"\033[{color_code}m┌{bar}┐\n│  {label:<{width-2}}│\n└{bar}┘\033[0m"


def ask(
    prompt: str,
    top_k: int = 1,
    verbose: bool = True,
) -> str:
    """
    1. Embed the prompt and search the Vector DB for the most relevant doc.
    2. Check the Redis semantic cache.
       → HIT : return cached response immediately (no LLM call).
       → MISS: call Groq, store result tagged with the retrieved doc_id.
    """
    if verbose:
        print(f"\n{'═'*64}")
        print(f"  QUERY: {prompt}")
        print(f"{'═'*64}")

    # ── Step 1: retrieve most-relevant doc from Vector DB ────────────
    query_vec = vectorizer.embed(prompt)
    results   = vector_db.search(COLLECTION, query_vec, top_k=top_k)

    source_id = None
    context   = ""
    if results:
        best     = results[0]
        source_id = best["doc_id"]
        context   = best["text"]
        if verbose:
            print(f"  VectorDB → {source_id}  (score={best['_score']:.3f})")
            print(f"  Context  : {context[:80]}…")
    else:
        if verbose:
            print("  VectorDB → no results found")

    # ── Step 2: check Redis semantic cache ───────────────────────────
    cached = manager.check(prompt)

    if cached is not None:
        if verbose:
            print()
            print(_banner("[CACHE HIT] — returning stored response (no LLM call)", "92"))
            print(f"\n  Response:\n{cached}\n")
        return cached

    # ── Step 3: cache miss → call Groq ───────────────────────────────
    if verbose:
        print()
        print(_banner("[CACHE MISS] — calling Groq LLM…", "91"))

    augmented_prompt = prompt
    if context:
        augmented_prompt = (
            f"Use the following context to answer the question.\n\n"
            f"Context:\n{context}\n\n"
            f"Question: {prompt}"
        )

    t0 = time.time()
    completion = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": augmented_prompt}],
        max_tokens=256,
    )
    elapsed  = time.time() - t0
    response = completion.choices[0].message.content

    if verbose:
        print(f"  Groq responded in {elapsed:.2f}s")

    # ── Step 4: store in Redis cache ─────────────────────────────────
    sid = source_id or "general"
    manager.store(prompt=prompt, response=response, source_id=sid)

    if verbose:
        print(f"  Stored in cache under source_id='{sid}'")
        print(f"\n  Response:\n{response}\n")

    return response


print("ask() helper ready")

ask() helper ready


In [10]:
%%time
# ── First call → CACHE MISS (Groq is called) ─────────────────────────
_ = ask("What causes climate change?")


════════════════════════════════════════════════════════════════
  QUERY: What causes climate change?
════════════════════════════════════════════════════════════════
  VectorDB → doc:climate  (score=0.619)
  Context  : Climate change is primarily driven by human emissions of greenhouse gases like C…

┌──────────────────────────────────────────────────────────────┐
│  [CACHE HIT] — returning stored response (no LLM call)       │
└──────────────────────────────────────────────────────────────┘

  Response:
**Climate change** – the long‑term shift in Earth’s weather patterns and temperatures – is driven by a combination of human‑made (anthropogenic) forces and natural processes.  The overwhelming consensus from the scientific community is that the recent warming trend is dominated by human activities.

---

## 1. Anthropogenic (Human‑Made) Drivers  

| Driver | How it Works | Typical Sources |
|--------|--------------|-----------------|
| **Greenhouse Gas (GHG) Emissions** | Gases that 

In [11]:
%%time
# ── Semantically similar → CACHE HIT ─────────────────────────────────
_ = ask("What are the main drivers of global warming?")


════════════════════════════════════════════════════════════════
  QUERY: What are the main drivers of global warming?
════════════════════════════════════════════════════════════════
  VectorDB → doc:climate  (score=0.609)
  Context  : Climate change is primarily driven by human emissions of greenhouse gases like C…

┌──────────────────────────────────────────────────────────────┐
│  [CACHE HIT] — returning stored response (no LLM call)       │
└──────────────────────────────────────────────────────────────┘

  Response:
**Climate change** – the long‑term shift in Earth’s weather patterns and temperatures – is driven by a combination of human‑made (anthropogenic) forces and natural processes.  The overwhelming consensus from the scientific community is that the recent warming trend is dominated by human activities.

---

## 1. Anthropogenic (Human‑Made) Drivers  

| Driver | How it Works | Typical Sources |
|--------|--------------|-----------------|
| **Greenhouse Gas (GHG) Emission

In [13]:
%%time
# ── Another topic — first call = MISS ────────────────────────────────
_ = ask("Why is C++ a popular programming language?")


════════════════════════════════════════════════════════════════
  QUERY: Why is C++ a popular programming language?
════════════════════════════════════════════════════════════════
  VectorDB → doc:python  (score=0.340)
  Context  : Python is a high-level, interpreted programming language known for its clean syn…

┌──────────────────────────────────────────────────────────────┐
│  [CACHE MISS] — calling Groq LLM…                            │
└──────────────────────────────────────────────────────────────┘
  Groq responded in 1.95s
  Stored in cache under source_id='doc:python'

  Response:
C++ remains popular because it blends high‑level productivity with low‑level control, giving developers the tools needed for a wide range of demanding applications. Here are the key factors that keep C++ in the spotlight:

| Why it

CPU times: user 834 ms, sys: 112 ms, total: 946 ms
Wall time: 2.21 s


In [14]:
%%time
# ── Repeat (near-identical) → HIT ────────────────────────────────────
_ = ask("Why do developers love C++?")


════════════════════════════════════════════════════════════════
  QUERY: Why do developers love C++?
════════════════════════════════════════════════════════════════
  VectorDB → doc:python  (score=0.278)
  Context  : Python is a high-level, interpreted programming language known for its clean syn…

┌──────────────────────────────────────────────────────────────┐
│  [CACHE HIT] — returning stored response (no LLM call)       │
└──────────────────────────────────────────────────────────────┘

  Response:
C++ remains popular because it blends high‑level productivity with low‑level control, giving developers the tools needed for a wide range of demanding applications. Here are the key factors that keep C++ in the spotlight:

| Why it

CPU times: user 504 ms, sys: 1.32 ms, total: 505 ms
Wall time: 66 ms


In [16]:
%%time
_ = ask("How do transformer neural networks work?")


════════════════════════════════════════════════════════════════
  QUERY: How do transformer neural networks work?
════════════════════════════════════════════════════════════════
  VectorDB → doc:transformers  (score=0.612)
  Context  : Transformer neural networks use a self-attention mechanism to process sequences …

┌──────────────────────────────────────────────────────────────┐
│  [CACHE HIT] — returning stored response (no LLM call)       │
└──────────────────────────────────────────────────────────────┘

  Response:
## Transformers: The “Attention‑First” Revolution in Neural Networks

Transformers are a class of deep learning models that have become the work‑horse for natural language processing (NLP), computer vision, speech, and many other domains. Their name comes from the *self‑attention* mechanism that lets the model “pay attention” to different parts of its input when computing each output token. Below is a tour‑style explanation that covers:

1. **Why we needed transform

## 6. Update a Document in the Vector DB → Invalidate Cache

When source content changes the cached responses are stale.  
This section shows how to:
1. Update the document text in the Vector DB.
2. Call `CacheInvalidator.delete_by_tag(doc_id)` to remove stale cache entries.
3. Re-ask the same question — now a **MISS** → fresh Groq response stored.

In [17]:
# ── Show the document BEFORE update ─────────────────────────────────
TARGET_DOC = "doc:climate"

doc = vector_db.get(COLLECTION, TARGET_DOC)
print(f"Document  : {TARGET_DOC}")
print(f"Updated at: {doc['updated_at']}")
print(f"\nText (before):\n{doc['text']}")

Document  : doc:climate
Updated at: 2026-05-06 03:00:52.466920

Text (before):
Climate change is primarily driven by human emissions of greenhouse gases like CO2 from fossil fuels, deforestation, and industrial processes. These gases trap heat in the atmosphere, raising global temperatures.Climate change is also reflected in a wide range of measurable environmental indicators beyond greenhouse gas concentrations. Global average surface temperatures have risen significantly since the late 19th century, with the most rapid warming occurring in recent decades. This warming is closely linked to the increased frequency and intensity of extreme weather events such as heatwaves, heavy rainfall, droughts, and tropical storms. Ocean temperatures have also increased, contributing to coral bleaching and the disruption of marine ecosystems, while thermal expansion and melting glaciers and ice sheets have driven a steady rise in sea levels. Arctic sea ice extent has declined sharply, particularly d

In [18]:
# ── Update the document in the Vector DB ────────────────────────────
UPDATED_TEXT = (
    "Climate change is caused by both natural factors and human activities. "
    "The dominant driver since the Industrial Revolution is the burning of "
    "fossil fuels (coal, oil, natural gas), releasing CO2 and methane. "
    "Deforestation, agriculture, and cement production are secondary contributors. "
    "The resulting greenhouse effect raises average global temperatures, "
    "intensifying extreme weather events and sea-level rise."
)

new_embedding = vectorizer.embed(UPDATED_TEXT)

vector_db.upsert(
    COLLECTION,
    {
        "doc_id":    TARGET_DOC,
        "text":      UPDATED_TEXT,
        "embedding": new_embedding,
        "metadata":  {"version": 2},
    },
)

updated = vector_db.get(COLLECTION, TARGET_DOC)
print(f"✅ Vector DB updated — new timestamp: {updated['updated_at']}")
print(f"\nNew text:\n{UPDATED_TEXT}")

✅ Vector DB updated — new timestamp: 2026-05-06 03:04:21.522253

New text:
Climate change is caused by both natural factors and human activities. The dominant driver since the Industrial Revolution is the burning of fossil fuels (coal, oil, natural gas), releasing CO2 and methane. Deforestation, agriculture, and cement production are secondary contributors. The resulting greenhouse effect raises average global temperatures, intensifying extreme weather events and sea-level rise.


In [19]:
# ── Invalidate stale cache entries for this document ─────────────────
print(f"Invalidating cache for source_id='{TARGET_DOC}'…")

deleted = invalidator.delete_by_tag(TARGET_DOC)

if deleted > 0:
    print(_banner(f"🗑️  INVALIDATED {deleted} cache entry/entries for '{TARGET_DOC}'", "93"))
else:
    print(f"  No cache entries found for '{TARGET_DOC}' (already empty).")

Invalidating cache for source_id='doc:climate'…
┌──────────────────────────────────────────────────────────────┐
│  🗑️  INVALIDATED 1 cache entry/entries for 'doc:climate'     │
└──────────────────────────────────────────────────────────────┘


In [20]:
# ── Re-ask — cache is gone → fresh Groq call with updated context ────
_ = ask("What causes climate change?")


════════════════════════════════════════════════════════════════
  QUERY: What causes climate change?
════════════════════════════════════════════════════════════════
  VectorDB → doc:climate  (score=0.769)
  Context  : Climate change is caused by both natural factors and human activities. The domin…

┌──────────────────────────────────────────────────────────────┐
│  [CACHE MISS] — calling Groq LLM…                            │
└──────────────────────────────────────────────────────────────┘
  Groq responded in 0.58s
  Stored in cache under source_id='doc:climate'

  Response:
Climate change is driven by a combination of natural factors and human activities. The main human‑induced cause since the Industrial Revolution has been the burning of fossil fuels—coal, oil, and natural gas—which releases large amounts of carbon dioxide (CO₂) and methane (CH₄) into the atmosphere. These gases strengthen the greenhouse effect, raising global temperatures. Other significant human contributions i

In [21]:
# ── Now the new response is cached → HIT ─────────────────────────────
_ = ask("What are the main drivers of global warming?")


════════════════════════════════════════════════════════════════
  QUERY: What are the main drivers of global warming?
════════════════════════════════════════════════════════════════
  VectorDB → doc:climate  (score=0.665)
  Context  : Climate change is caused by both natural factors and human activities. The domin…

┌──────────────────────────────────────────────────────────────┐
│  [CACHE MISS] — calling Groq LLM…                            │
└──────────────────────────────────────────────────────────────┘
  Groq responded in 0.42s
  Stored in cache under source_id='doc:climate'

  Response:
The main drivers of global warming are:

1. **Burning of fossil fuels (coal, oil, natural gas)** – releases large amounts of CO₂ and methane, and has been the dominant force since the Industrial Revolution.  
2. **Deforestation** – reduces the planet’s capacity to absorb CO₂ and often releases stored carbon.  
3. **Agriculture** – contributes methane (from livestock and rice paddies) and nitrou

## 7. Inspect Redis Cache Contents

Browse every entry currently stored in the Redis index —  
prompt, source_id tag, and similarity-vector dimension count.

In [22]:
import redis as _redis_lib

r = _redis_lib.from_url(REDIS_URL, decode_responses=False)

pattern = f"{INDEX_NAME}:entry:*"
keys    = r.keys(pattern)

print(f"Redis index  : '{INDEX_NAME}'")
print(f"Cache entries: {len(keys)}")
print()

if not keys:
    print("  (empty — run cells in Section 5 first)")
else:
    for i, key in enumerate(sorted(keys), 1):
        key_str = key.decode() if isinstance(key, bytes) else key
        try:
            doc = r.json().get(key_str, "$")
            if doc and len(doc) > 0:
                entry    = doc[0]
                prompt   = entry.get("prompt", "")[:80]
                source   = entry.get("source_id", "—")
                vec_len  = len(entry.get("prompt_vector", []))
                response = entry.get("response", "")[:60]
                print(f"  [{i:02d}] key       : {key_str}")
                print(f"       source_id : {source}")
                print(f"       prompt    : {prompt}…")
                print(f"       response  : {response}…")
                print(f"       vector dim: {vec_len}")
                print()
        except Exception as e:
            print(f"  [{i:02d}] {key_str} — could not read ({e})")

Redis index  : 'rag_cache'
Cache entries: 7

  [01] key       : rag_cache:entry:1cc8cfb67585bf30e634c0b2c9d61f889423913cbcba6c53cb44b8b3da731de2
       source_id : doc:transformers
       prompt    : Explain the concept of transformer neural networks.…
       response  : ## Transformers: The “Attention‑First” Revolution in Neural …
       vector dim: 384

  [02] key       : rag_cache:entry:7b8baa86c6f6980e5652be6e5d6f830d19840f731acfa039df2856d287b3cfd9
       source_id : doc:python
       prompt    : Why is C++ a popular programming language?…
       response  : C++ remains popular because it blends high‑level productivit…
       vector dim: 384

  [03] key       : rag_cache:entry:9c2400054219022b9d0ce951c6cb6e5ba3947c5790ec79ee76bc49ad243530e3
       source_id : doc:python
       prompt    : Why is Python a popular programming language?…
       response  : Python’s popularity isn’t a mystery—it’s a combination of de…
       vector dim: 384

  [04] key       : rag_cache:entry:a3ad209e

In [23]:
# ── Show a summary grouped by source_id ─────────────────────────────
from collections import defaultdict

by_source = defaultdict(list)

for key in keys:
    key_str = key.decode() if isinstance(key, bytes) else key
    try:
        doc = r.json().get(key_str, "$")
        if doc and len(doc) > 0:
            entry  = doc[0]
            source = entry.get("source_id", "unknown")
            prompt = entry.get("prompt", "")[:60]
            by_source[source].append(prompt)
    except Exception:
        pass

print(f"Cache entries grouped by source_id ({len(by_source)} sources):\n")
for source, prompts in sorted(by_source.items()):
    print(f"  {source}  ({len(prompts)} entr{'y' if len(prompts)==1 else 'ies'})")
    for p in prompts:
        print(f"    • {p}…")
    print()

Cache entries grouped by source_id (4 sources):

  doc:climate  (2 entries)
    • What causes climate change?…
    • What are the main drivers of global warming?…

  doc:python  (3 entries)
    • What is Python used for?…
    • Why is C++ a popular programming language?…
    • Why is Python a popular programming language?…

  doc:redis  (1 entry)
    • What is Redis?…

  doc:transformers  (1 entry)
    • Explain the concept of transformer neural networks.…



In [24]:
# ── Show current state of the Vector DB ─────────────────────────────
all_docs = vector_db.list_all(COLLECTION)

print(f"Vector DB collection '{COLLECTION}' — {len(all_docs)} documents:\n")
for doc in all_docs:
    meta = doc.get("metadata") or {}
    print(f"  doc_id    : {doc['doc_id']}")
    print(f"  updated_at: {doc['updated_at']}")
    print(f"  metadata  : {meta}")
    print(f"  text      : {doc['text'][:80]}…")
    print(f"  embedding : [{doc['embedding'][0]:.4f}, {doc['embedding'][1]:.4f}, … ] dim={len(doc['embedding'])}")
    print()

Vector DB collection 'knowledge_base' — 5 documents:

  doc_id    : doc:python
  updated_at: 2026-05-06 03:00:52.467074
  metadata  : {}
  text      : Python is a high-level, interpreted programming language known for its clean syn…
  embedding : [-0.0543, -0.0353, … ] dim=384

  doc_id    : doc:redis
  updated_at: 2026-05-06 03:00:52.467137
  metadata  : {}
  text      : Redis is an open-source, in-memory data structure store used as a database, cach…
  embedding : [-0.0015, -0.0183, … ] dim=384

  doc_id    : doc:transformers
  updated_at: 2026-05-06 03:00:52.467254
  metadata  : {}
  text      : Transformer neural networks use a self-attention mechanism to process sequences …
  embedding : [-0.1504, -0.0420, … ] dim=384

  doc_id    : doc:rag
  updated_at: 2026-05-06 03:00:52.467292
  metadata  : {}
  text      : Retrieval-Augmented Generation (RAG) enhances LLM responses by retrieving releva…
  embedding : [-0.0890, 0.0028, … ] dim=384

  doc_id    : doc:climate
  updated_at: 2026-

## 8. Full Cleanup

Drops the RediSearch index, deletes all cached documents, and clears the in-memory Vector DB.  
Run this cell when you are done testing.

In [25]:
# Drop Redis index + all documents
manager.delete_index(drop_documents=True)
print("✅ Redis index and all cached documents deleted.")

# Clear the in-memory Vector DB
vector_db._store.clear()
print("✅ In-memory Vector DB cleared.")
print("\nAll done. Re-run from Section 3 to start fresh.")

✅ Redis index and all cached documents deleted.
✅ In-memory Vector DB cleared.

All done. Re-run from Section 3 to start fresh.
